# Stage 2 | Can Native WorldGym Evaluate Our ACT Policies?

**Stage 1 trained two ACT policies that can pick up a tube. Here we test whether native WorldGym can provide the future observations needed to evaluate them.**

The main finding is negative: in this implementation, the native observation loop did not preserve successful pickup behavior. Read the actual frames below before interpreting the numbers. No Hybrid/CAD-generated frames are used here.

## 1 What ACT sees, and what we compare

ACT takes **Front RGB + Wrist RGB + a 7D joint state** and predicts an action chunk (arm-joint targets and gripper targets). Front shows the overall arm and rack; Wrist shows the local relationship between the fingers and the tube cap. Both camera views matter to the next decision.

Both ACT checkpoints stay frozen. Each policy starts from the same initial images and robot state in two separate loops:

| Reference loop | Native model loop |
|---|---|
| ACT chooses an action chunk | The same ACT chooses an action chunk |
| MuJoCo executes the actions | WorldGym predicts future camera observations from the actions |
| ACT reads new simulator images and joint state | ACT reads generated images and separately predicted joint state |
| Repeat, then apply the official success check | Repeat, then replay the resulting full action sequence in MuJoCo |

Only the reset observations initialize the native loop. Future real images, joint states and controls are not fed into it. Once observations differ, subsequent ACT actions can differ too.

The final physical replay checks whether the **actions produced inside the native loop** actually pick up the tube. It is not an extra feedback source for that loop, and it is not a calibrated success probability.

## 2 Read both camera views

These are **saved experimental frames**, not AI-generated teaching illustrations. We keep the same example selection: the smallest shared seed, **62259**, and fixed times **0, 10 and 20 seconds**. No new trial was selected to make the difference look larger.

Each figure has four rows: **MuJoCo Front, WorldGym Front, MuJoCo Wrist, WorldGym Wrist**. Compare adjacent rows within each camera view. The first column shows the starting observation; use the later columns to follow what happens to the arm, fingers and orange cap.

### 10-demo ACT

![10-demo ACT: paired MuJoCo and native WorldGym Front and Wrist frames at 0, 10 and 20 seconds.](media/figures/native_comparison_10demo.png)

**Read the Front pair:** at 10 and 20 seconds, the MuJoCo tube is above the rack. In the native frames, the orange cap remains near the rack and the arm has softened/ghosted structure.

**Now read the Wrist pair:** the MuJoCo cap is large and clearly visible between the fingers. In the native frames at those times, the orange cap is not clearly discernible between them, and the rack/background boundaries are softer. This view makes the missing local grasp relationship easier to see than Front alone.

[Watch the preserved Front + Wrist comparison video](media/demos/native_comparison_10demo.mp4).

### 80-demo ACT

![80-demo ACT: paired MuJoCo and native WorldGym Front and Wrist frames at 0, 10 and 20 seconds.](media/figures/native_comparison_80demo.png)

**Read the Front pair:** the MuJoCo frames show the raised tube, while the native frames retain the cap near the rack.

**Now read the Wrist pair:** at 10 and 20 seconds, MuJoCo shows a prominent orange cap between the fingers. The native frames instead show a small orange patch to the right of the fingers, with softened surrounding structure. The local configuration presented to ACT is visibly different.

[Watch the preserved Front + Wrist comparison video](media/demos/native_comparison_80demo.mp4).

**Why this matters:** ACT makes its next decision from these images and the joint state. A different apparent finger–cap relationship can change that decision. The relevant question is whether the observations preserve useful geometry and motion, not just whether the image looks sharp.

These are **separate closed loops**, so later actions may differ. Their same-time pictures are not a same-action prediction-error test. The images show observed differences; the table below tests the resulting actions. Together they document failure of the current combined loop, but do not isolate blur as its cause.


## 3 Did the resulting actions pick up the tube?

The archived diagnostic contains **ten common development starts per policy**. All twenty cases are included, including failures.

| Frozen policy | MuJoCo observation loop | Native-loop commands replayed in MuJoCo |
|---|---:|---:|
| 10-demo ACT | **8/10** | **0/10** |
| 80-demo ACT | **10/10** | **0/10** |

Both columns use the official pickup check: tube body height above 0.925 m and left-finger contact with the cap. The right column measures physical replay of model-loop actions, not a judgment that a generated image looks successful.

These are **development diagnostics**, separate from Stage 1's 50 starts and from the old Hybrid evaluator's 20-start test. Each native trajectory records 23 ACT calls, 1,130 controls and 114 frames. The cell below recounts the bundled original records; it does not run another experiment.

In [1]:
from briefing_utils import *
verify_assets()
check_stage_evidence()
native_evaluation_results()

Frozen policy,MuJoCo observation loop,Native-loop commands replayed in MuJoCo
10-demo ACT,8/10,0/10
80-demo ACT,10/10,0/10


## 4 What we conclude, and what comes next

**The two policies can pick up the tube with simulator observations, but their actions under the current native observation loop failed in this diagnostic. This implementation has not established reliable native WorldGym policy evaluation.**

The useful next step is to check prediction before adding optimization: give the model recorded real actions, compare predicted and real motion, then improve target-task adaptation and repeat the closed-loop test. More matched image–action–future-observation trajectories are a candidate remedy, not a demonstrated solution. The checkpoint was already task-adapted; this is not evidence that WorldGym necessarily fails in every unseen environment.

Until that observation loop is reliable, we do not treat model-based RL as an established next step. Separate Hybrid and model-based RL explorations remain in the [earlier project](https://github.com/Kevin11Kaikai/worldgym-autobio); they do not replace this native result.

**Evidence:** [all twenty outcomes](evidence/stage2_native_outcomes.csv), [copied original records and source hashes](evidence/stage2_source_records.json), and [diagnostic summary](evidence/stage2_native_summary.json). The four-state open-loop world-model repair study is a different experiment and is not used for this table.

**Reproduction:** [Scripts/README.md](Scripts/README.md) separates CPU record verification from GPU reruns and documents asset availability.

[Back to Stage 1](01_act_imitation_learning.ipynb)

A later notebook asks whether a latent model can be compared with its own simulator when no future images are generated: [03 · Latent paired evaluation on PickCube](03_latent_pickcube_policy_evaluation.ipynb). That comparison uses SAPIEN/PhysX and a different task. It does not change the 0/10 result above.

[Notebook 4](04_latent_tube_policy_evaluation.ipynb) then asks the same paired comparison for a frozen tube controller and a tube-specific latent model. It does not repair the native image loop or change the 0/10 result above.
